In [0]:
# ============================================================
# DAY 4 - DATABASE INGESTION
# BLOCK 1: DATABRICKS PROJECT SETUP
# ============================================================

from datetime import datetime

# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

CATALOG = "de_prac"
SCHEMA = "day04"
VOLUME = "data-files"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

BRONZE_PATH = f"{BASE_PATH}/bronze"
LOG_PATH = f"{BASE_PATH}/logs"
REPORT_PATH = f"{BASE_PATH}/reports"
QUARANTINE_PATH = f"{BASE_PATH}/quarantine"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.customers_bronze"


# ------------------------------------------------------------
# 2. CREATE REQUIRED FOLDERS
# ------------------------------------------------------------

folders = [
    BRONZE_PATH,
    LOG_PATH,
    REPORT_PATH,
    QUARANTINE_PATH
]

for folder in folders:
    dbutils.fs.mkdirs(folder)

print("Folders created successfully.")


# ------------------------------------------------------------
# 3. CREATE DATABRICKS PARAMETERS
# ------------------------------------------------------------

dbutils.widgets.removeAll()

dbutils.widgets.text(
    "requested_status",
    "active",
    "Customer Status"
)

dbutils.widgets.text(
    "jdbc_host",
    "",
    "PostgreSQL Host"
)

dbutils.widgets.text(
    "jdbc_port",
    "5432",
    "PostgreSQL Port"
)

dbutils.widgets.text(
    "jdbc_database",
    "",
    "Database Name"
)

dbutils.widgets.text(
    "jdbc_user",
    "",
    "Database User"
)

dbutils.widgets.text(
    "jdbc_password",
    "",
    "Database Password"
)


# ------------------------------------------------------------
# 4. READ PARAMETERS
# ------------------------------------------------------------

requested_status = dbutils.widgets.get("requested_status")

jdbc_host = dbutils.widgets.get("jdbc_host")
jdbc_port = dbutils.widgets.get("jdbc_port")
jdbc_database = dbutils.widgets.get("jdbc_database")
jdbc_user = dbutils.widgets.get("jdbc_user")
jdbc_password = dbutils.widgets.get("jdbc_password")


# ------------------------------------------------------------
# 5. BUILD JDBC CONFIGURATION
# ------------------------------------------------------------

jdbc_url = (
    f"jdbc:postgresql://{jdbc_host}:{jdbc_port}/{jdbc_database}"
    if jdbc_host and jdbc_database
    else None
)

jdbc_properties = {
    "user": jdbc_user,
    "password": jdbc_password,
    "driver": "org.postgresql.Driver"
}


# ------------------------------------------------------------
# 6. CREATE RUN ID
# ------------------------------------------------------------

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

run_bronze_path = f"{BRONZE_PATH}/run_{run_id}"
run_log_path = f"{LOG_PATH}/run_{run_id}"
run_report_path = f"{REPORT_PATH}/run_{run_id}"


# ------------------------------------------------------------
# 7. DISPLAY CONFIGURATION
# ------------------------------------------------------------

print("\n================ DAY 4 CONFIGURATION ================")

print(f"Catalog            : {CATALOG}")
print(f"Schema             : {SCHEMA}")
print(f"Volume             : {VOLUME}")

print(f"\nBase Path          : {BASE_PATH}")
print(f"Bronze Path        : {BRONZE_PATH}")
print(f"Log Path           : {LOG_PATH}")
print(f"Report Path        : {REPORT_PATH}")
print(f"Quarantine Path    : {QUARANTINE_PATH}")

print(f"\nBronze Table       : {BRONZE_TABLE}")
print(f"Requested Status   : {requested_status}")

print(f"\nRun ID             : {run_id}")
print(f"Run Bronze Path    : {run_bronze_path}")

if jdbc_url:
    print(f"\nJDBC URL           : {jdbc_url}")
else:
    print("\nJDBC connection details not configured yet.")

print("\nDay 4 Databricks setup completed successfully.")


# ------------------------------------------------------------
# 8. VERIFY VOLUME CONTENTS
# ------------------------------------------------------------

display(dbutils.fs.ls(BASE_PATH))

In [0]:
# ============================================================
# DAY 4 - BLOCK 3A
# FIND DATABRICKS OUTBOUND IP
# ============================================================

import urllib.request

try:
    databricks_outbound_ip = (
        urllib.request
        .urlopen("https://ifconfig.me", timeout=15)
        .read()
        .decode()
        .strip()
    )

    print("Databricks outbound IP:", databricks_outbound_ip)

except Exception as e:
    print("Outbound internet test failed.")
    print("If 'Verify identity' is still visible in Databricks, complete that first.")
    print("Error:", str(e))

In [0]:
# ============================================================
# DAY 4 - BLOCK 3 FIXED FOR SERVERLESS COMPUTE
# JDBC EXTRACTION -> DELTA MATERIALIZATION -> VALIDATION -> BRONZE
# ============================================================

from datetime import datetime
from pyspark.sql import functions as F


# ------------------------------------------------------------
# 1. READ PARAMETERS
# ------------------------------------------------------------

requested_status = dbutils.widgets.get("requested_status").strip()
jdbc_host = dbutils.widgets.get("jdbc_host").strip()
jdbc_port = dbutils.widgets.get("jdbc_port").strip()
jdbc_database = dbutils.widgets.get("jdbc_database").strip()
jdbc_user = dbutils.widgets.get("jdbc_user").strip()
jdbc_password = dbutils.widgets.get("jdbc_password")

CATALOG = "de_prac"
SCHEMA = "day04"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/data-files"
BRONZE_PATH = f"{BASE_PATH}/bronze"
LOG_PATH = f"{BASE_PATH}/logs"
REPORT_PATH = f"{BASE_PATH}/reports"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.customers_bronze"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")


# ------------------------------------------------------------
# 2. JDBC CONNECTION
# ------------------------------------------------------------

jdbc_url = (
    f"jdbc:postgresql://{jdbc_host}:{jdbc_port}/{jdbc_database}"
    "?sslmode=require"
)

jdbc_properties = {
    "user": jdbc_user,
    "password": jdbc_password,
    "driver": "org.postgresql.Driver"
}

print("============================================================")
print("DAY 4 - POSTGRESQL JDBC EXTRACTION")
print("============================================================")

print(f"Host             : {jdbc_host}")
print(f"Port             : {jdbc_port}")
print(f"Database         : {jdbc_database}")
print(f"User             : {jdbc_user}")
print(f"Requested Status : {requested_status}")
print(f"Run ID           : {run_id}")


# ------------------------------------------------------------
# 3. CONNECTION TEST
# ------------------------------------------------------------

print("\n[1] Testing PostgreSQL JDBC connection...")

connection_test_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table="(SELECT 1 AS connection_test) AS t",
        properties=jdbc_properties
    )
)

connection_test_df.collect()

print("JDBC CONNECTION SUCCESSFUL")


# ------------------------------------------------------------
# 4. VALIDATE REQUESTED STATUS
# ------------------------------------------------------------

allowed_statuses = ["active", "inactive", "suspended"]

if requested_status not in allowed_statuses:
    raise ValueError(
        f"Unsupported status: {requested_status}"
    )


# ------------------------------------------------------------
# 5. SOURCE QUERY
# ------------------------------------------------------------

source_query = f"""
(
    SELECT
        customer_id,
        first_name,
        last_name,
        email,
        city,
        status
    FROM customer_source
    WHERE status = '{requested_status}'
) AS customer_extract
"""


# ------------------------------------------------------------
# 6. READ POSTGRESQL -> SPARK DATAFRAME
# ------------------------------------------------------------

print("\n[2] Extracting customers from PostgreSQL...")

source_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table=source_query,
        properties=jdbc_properties
    )
)


# ------------------------------------------------------------
# 7. ADD INGESTION METADATA
# ------------------------------------------------------------

extract_df = (
    source_df
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_system",
        F.lit("azure_postgresql")
    )
    .withColumn(
        "run_id",
        F.lit(run_id)
    )
)


# ------------------------------------------------------------
# 8. MATERIALIZE THE EXTRACT
#
# Instead of .cache(), write it once to Delta.
# This works on Databricks Serverless.
# ------------------------------------------------------------

run_extract_path = (
    f"{BRONZE_PATH}/run_{run_id}_status-{requested_status}"
)

print("\n[3] Materializing JDBC extract to Delta...")

(
    extract_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(run_extract_path)
)

print("Extract materialized successfully.")
print(f"Path: {run_extract_path}")


# ------------------------------------------------------------
# 9. READ MATERIALIZED DATA
#
# All validations below use Delta instead of hitting PostgreSQL
# repeatedly.
# ------------------------------------------------------------

customer_df = (
    spark.read
    .format("delta")
    .load(run_extract_path)
)


# ------------------------------------------------------------
# 10. EXTRACT COUNT
# ------------------------------------------------------------

extract_count = customer_df.count()

print("\n[4] Extract count")
print(f"Extracted record count: {extract_count:,}")


# ------------------------------------------------------------
# 11. SCHEMA VALIDATION
# ------------------------------------------------------------

expected_business_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "city",
    "status"
]

actual_business_columns = [
    c for c in customer_df.columns
    if c not in [
        "ingestion_timestamp",
        "source_system",
        "run_id"
    ]
]

schema_valid = (
    actual_business_columns == expected_business_columns
)

print("\n[5] Schema validation")
print("Expected :", expected_business_columns)
print("Actual   :", actual_business_columns)
print("Passed   :", schema_valid)


# ------------------------------------------------------------
# 12. ACTIVE / REQUESTED STATUS VALIDATION
# ------------------------------------------------------------

invalid_status_count = (
    customer_df
    .filter(
        F.col("status").isNull() |
        (F.col("status") != requested_status)
    )
    .count()
)

status_validation_passed = (
    invalid_status_count == 0
)

print("\n[6] Status validation")
print(f"Requested status : {requested_status}")
print(f"Invalid rows     : {invalid_status_count}")
print(f"Passed           : {status_validation_passed}")


# ------------------------------------------------------------
# 13. DUPLICATE BUSINESS KEYS
# ------------------------------------------------------------

duplicate_df = (
    customer_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_df.count()

print("\n[7] Duplicate business-key validation")
print(
    f"Duplicate customer IDs: "
    f"{duplicate_key_count}"
)


# ------------------------------------------------------------
# 14. NULL STATUS VALIDATION
# ------------------------------------------------------------

null_status_count = (
    customer_df
    .filter(F.col("status").isNull())
    .count()
)

print("\n[8] NULL status validation")
print(
    f"NULL statuses in extracted data: "
    f"{null_status_count}"
)


# ------------------------------------------------------------
# 15. WRITE FINAL BRONZE TABLE
# ------------------------------------------------------------

print("\n[9] Writing Bronze Delta table...")

(
    customer_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(
    f"Bronze table created: "
    f"{BRONZE_TABLE}"
)


# ------------------------------------------------------------
# 16. SOURCE -> BRONZE RECONCILIATION
# ------------------------------------------------------------

bronze_count = (
    spark.table(BRONZE_TABLE)
    .count()
)

print("\n[10] Reconciliation")

print(
    f"Extract count : "
    f"{extract_count:,}"
)

print(
    f"Bronze count  : "
    f"{bronze_count:,}"
)

if bronze_count != extract_count:
    raise Exception(
        "RECONCILIATION FAILED"
    )

print("Source-to-Bronze reconciliation PASSED")


# ------------------------------------------------------------
# 17. EXECUTION LOG
# ------------------------------------------------------------

log_text = f"""
DAY 4 - DATABASE INGESTION

Run ID:
{run_id}

Source system:
Azure Database for PostgreSQL

Source database:
{jdbc_database}

Source table:
customer_source

Requested status:
{requested_status}

Extract count:
{extract_count}

Duplicate business keys:
{duplicate_key_count}

NULL statuses:
{null_status_count}

Schema validation:
{schema_valid}

Status validation:
{status_validation_passed}

Bronze count:
{bronze_count}

Bronze table:
{BRONZE_TABLE}

Extract path:
{run_extract_path}

Result:
SUCCESS
"""

log_file = (
    f"{LOG_PATH}/day04_{run_id}.txt"
)

dbutils.fs.put(
    log_file,
    log_text,
    overwrite=True
)

print(
    f"\nExecution log written: "
    f"{log_file}"
)


# ------------------------------------------------------------
# 18. VALIDATION REPORT
# ------------------------------------------------------------

report_data = [
    (
        run_id,
        requested_status,
        extract_count,
        duplicate_key_count,
        null_status_count,
        schema_valid,
        status_validation_passed,
        bronze_count
    )
]

report_columns = [
    "run_id",
    "requested_status",
    "extract_count",
    "duplicate_key_count",
    "null_status_count",
    "schema_valid",
    "status_validation_passed",
    "bronze_count"
]

report_df = spark.createDataFrame(
    report_data,
    report_columns
)

report_path = (
    f"{REPORT_PATH}/day04_{run_id}"
)

(
    report_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(report_path)
)

print(
    f"Validation report written: "
    f"{report_path}"
)


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n============================================================")
print("DAY 4 JDBC INGESTION COMPLETED SUCCESSFULLY")
print("============================================================")

print(f"""
Run ID              : {run_id}
Requested status    : {requested_status}
Extracted rows      : {extract_count:,}
Duplicate keys      : {duplicate_key_count}
NULL statuses       : {null_status_count}
Schema valid        : {schema_valid}
Status valid        : {status_validation_passed}
Bronze rows         : {bronze_count:,}
Bronze table        : {BRONZE_TABLE}
""")


# ------------------------------------------------------------
# 20. DISPLAY SAMPLE
# ------------------------------------------------------------

display(
    customer_df
    .orderBy("customer_id")
    .limit(20)
)

print("Duplicate business keys:")

display(
    duplicate_df
    .orderBy("customer_id")
)

In [0]:
# ============================================================
# DAY 4 OF 15
# BLOCK 4 - FINAL EDGE CASE TESTS + TRAINER EVIDENCE
# ============================================================

from datetime import datetime
from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

jdbc_host = dbutils.widgets.get("jdbc_host").strip()
jdbc_port = dbutils.widgets.get("jdbc_port").strip()
jdbc_database = dbutils.widgets.get("jdbc_database").strip()
jdbc_user = dbutils.widgets.get("jdbc_user").strip()
jdbc_password = dbutils.widgets.get("jdbc_password")

CATALOG = "de_prac"
SCHEMA = "day04"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/data-files"
REPORT_PATH = f"{BASE_PATH}/reports"
LOG_PATH = f"{BASE_PATH}/logs"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.customers_bronze"
TEST_TABLE = f"{CATALOG}.{SCHEMA}.day04_test_results"

test_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")


jdbc_url = (
    f"jdbc:postgresql://{jdbc_host}:{jdbc_port}/{jdbc_database}"
    "?sslmode=require&connectTimeout=5"
)

jdbc_properties = {
    "user": jdbc_user,
    "password": jdbc_password,
    "driver": "org.postgresql.Driver"
}


print("=" * 65)
print("DAY 4 - FINAL ACCEPTANCE & EDGE CASE TESTS")
print("=" * 65)

print(f"Test Run ID : {test_run_id}")
print(f"Database    : {jdbc_database}")
print(f"Bronze      : {BRONZE_TABLE}")


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

ALLOWED_STATUSES = {
    "active",
    "inactive",
    "suspended"
}


def read_source_table(
    url=jdbc_url,
    properties=jdbc_properties
):
    return (
        spark.read
        .jdbc(
            url=url,
            table="customer_source",
            properties=properties
        )
    )


def extract_by_status(status):
    """
    Runtime status parameter.

    No business value is concatenated into a SQL string.
    Spark receives the requested status as a column expression.
    """

    if status not in ALLOWED_STATUSES:
        raise ValueError(
            f"Unsupported status: {status}"
        )

    return (
        read_source_table()
        .filter(
            F.col("status") == F.lit(status)
        )
    )


test_results = []


def record_test(
    test_name,
    expected,
    actual,
    passed,
    details
):
    test_results.append(
        (
            test_run_id,
            test_name,
            str(expected),
            str(actual),
            "PASS" if passed else "FAIL",
            details
        )
    )

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{test_name} | "
        f"Expected={expected} | "
        f"Actual={actual}"
    )


# ============================================================
# 3. JDBC CONNECTION TEST
# ============================================================

print("\n[1] JDBC CONNECTION")

try:

    (
        spark.read
        .jdbc(
            url=jdbc_url,
            table="(SELECT 1 AS connection_test) AS t",
            properties=jdbc_properties
        )
        .collect()
    )

    record_test(
        "JDBC connection",
        "Successful",
        "Successful",
        True,
        "Databricks connected to Azure PostgreSQL."
    )

except Exception as e:

    record_test(
        "JDBC connection",
        "Successful",
        "Failed",
        False,
        str(e).splitlines()[0]
    )

    raise


# ============================================================
# 4. SOURCE TOTAL COUNT
# ============================================================

print("\n[2] SOURCE RECORD COUNT")

source_count = (
    read_source_table()
    .count()
)

record_test(
    "Source total count",
    100005,
    source_count,
    source_count == 100005,
    "100,000 generated rows plus 5 intentional duplicates."
)


# ============================================================
# 5. ACTIVE-ONLY EXTRACTION
# ============================================================

print("\n[3] ACTIVE-ONLY EXTRACTION")

active_df = extract_by_status("active")

active_count = active_df.count()

active_distinct_statuses = [
    row["status"]
    for row in
    active_df
    .select("status")
    .distinct()
    .collect()
]

active_filter_passed = (
    active_count == 70005
    and active_distinct_statuses == ["active"]
)

record_test(
    "Active-only extraction",
    "70005 active rows only",
    (
        f"{active_count} rows, "
        f"statuses={active_distinct_statuses}"
    ),
    active_filter_passed,
    "Validates the business status filter."
)


# ============================================================
# 6. ZERO-MATCH TEST
# ============================================================

print("\n[4] ZERO-MATCH EXTRACTION")

zero_df = extract_by_status("suspended")

zero_count = zero_df.count()

record_test(
    "Zero matches",
    0,
    zero_count,
    zero_count == 0,
    "Requested status 'suspended' does not exist in source."
)


# ============================================================
# 7. DUPLICATE BUSINESS-KEY TEST
# ============================================================

print("\n[5] DUPLICATE BUSINESS KEYS")

duplicate_df = (
    read_source_table()
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_key_count = duplicate_df.count()

record_test(
    "Duplicate business keys",
    5,
    duplicate_key_count,
    duplicate_key_count == 5,
    "Five customer_id values were intentionally duplicated."
)


# ============================================================
# 8. NULL STATUS TEST
# ============================================================

print("\n[6] NULL STATUS")

source_null_status_count = (
    read_source_table()
    .filter(
        F.col("status").isNull()
    )
    .count()
)

record_test(
    "Source NULL status",
    10000,
    source_null_status_count,
    source_null_status_count == 10000,
    "NULL-status rows exist in the source for edge-case testing."
)


# ============================================================
# 9. NULL STATUS MUST NOT ENTER ACTIVE EXTRACT
# ============================================================

active_null_count = (
    active_df
    .filter(
        F.col("status").isNull()
    )
    .count()
)

record_test(
    "NULL excluded from active extract",
    0,
    active_null_count,
    active_null_count == 0,
    "NULL does not satisfy the active status filter."
)


# ============================================================
# 10. DUPLICATES SHOULD REMAIN IN EXTRACTION
# ============================================================

active_duplicate_count = (
    active_df
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

record_test(
    "Duplicate keys preserved during ingestion",
    5,
    active_duplicate_count,
    active_duplicate_count == 5,
    (
        "Day 4 performs extraction. "
        "Duplicate remediation belongs to validation/cleaning."
    )
)


# ============================================================
# 11. BRONZE COUNT RECONCILIATION
# ============================================================

print("\n[7] BRONZE RECONCILIATION")

bronze_count = (
    spark.table(BRONZE_TABLE)
    .count()
)

record_test(
    "Source-to-Bronze reconciliation",
    active_count,
    bronze_count,
    bronze_count == active_count,
    "Active source extract count matches Bronze table."
)


# ============================================================
# 12. BRONZE STATUS VALIDATION
# ============================================================

bronze_invalid_status_count = (
    spark.table(BRONZE_TABLE)
    .filter(
        F.col("status").isNull()
        |
        (F.col("status") != "active")
    )
    .count()
)

record_test(
    "Bronze active-only validation",
    0,
    bronze_invalid_status_count,
    bronze_invalid_status_count == 0,
    "Bronze contains active customers only."
)


# ============================================================
# 13. WRONG-CREDENTIAL TEST
# ============================================================

print("\n[8] WRONG CREDENTIAL TEST")

wrong_properties = dict(jdbc_properties)

wrong_properties["password"] = (
    "__INTENTIONALLY_WRONG_PASSWORD__"
)

wrong_credentials_failed = False

wrong_credentials_message = ""

try:

    (
        spark.read
        .jdbc(
            url=jdbc_url,
            table="(SELECT 1 AS test) AS t",
            properties=wrong_properties
        )
        .collect()
    )

except Exception as e:

    wrong_credentials_failed = True

    wrong_credentials_message = (
        str(e)
        .splitlines()[0][:300]
    )


record_test(
    "Wrong credentials",
    "Connection rejected",
    (
        "Connection rejected"
        if wrong_credentials_failed
        else "Unexpected connection success"
    ),
    wrong_credentials_failed,
    wrong_credentials_message
)


# ============================================================
# 14. DATABASE UNAVAILABLE TEST
# ============================================================

print("\n[9] DATABASE UNAVAILABLE TEST")

unavailable_url = (
    f"jdbc:postgresql://{jdbc_host}:5433/"
    f"{jdbc_database}"
    "?sslmode=require&connectTimeout=5"
)

database_unavailable_failed = False

database_unavailable_message = ""

try:

    (
        spark.read
        .jdbc(
            url=unavailable_url,
            table="(SELECT 1 AS test) AS t",
            properties=jdbc_properties
        )
        .collect()
    )

except Exception as e:

    database_unavailable_failed = True

    database_unavailable_message = (
        str(e)
        .splitlines()[0][:300]
    )


record_test(
    "Database unavailable",
    "Connection failure",
    (
        "Connection failure"
        if database_unavailable_failed
        else "Unexpected connection success"
    ),
    database_unavailable_failed,
    database_unavailable_message
)


# ============================================================
# 15. SCHEMA VALIDATION
# ============================================================

print("\n[10] SCHEMA VALIDATION")

expected_business_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "city",
    "status"
]

actual_business_columns = [
    c
    for c in spark.table(BRONZE_TABLE).columns
    if c not in {
        "ingestion_timestamp",
        "source_system",
        "run_id"
    }
]

schema_passed = (
    actual_business_columns
    == expected_business_columns
)

record_test(
    "Bronze schema",
    expected_business_columns,
    actual_business_columns,
    schema_passed,
    "Validates five customer columns plus status."
)


# ============================================================
# 16. CREATE TEST RESULTS DATAFRAME
# ============================================================

test_schema = [
    "test_run_id",
    "test_name",
    "expected",
    "actual",
    "result",
    "details"
]

test_df = spark.createDataFrame(
    test_results,
    test_schema
)


# ============================================================
# 17. SAVE TEST RESULTS AS DELTA TABLE
# ============================================================

(
    test_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TEST_TABLE)
)


# ============================================================
# 18. SAVE TRAINER REPORT AS CSV
# ============================================================

trainer_report_path = (
    f"{REPORT_PATH}/"
    f"trainer_review_{test_run_id}"
)

(
    test_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(trainer_report_path)
)


# ============================================================
# 19. FINAL PASS / FAIL
# ============================================================

failed_tests = [
    row
    for row in test_results
    if row[4] == "FAIL"
]

passed_count = (
    len(test_results)
    - len(failed_tests)
)

total_tests = len(test_results)


# ============================================================
# 20. CREATE TRAINER SUMMARY
# ============================================================

trainer_summary = f"""
============================================================
DAY 4 OF 15 - TRAINER REVIEW SUMMARY
============================================================

Story:
Database ingestion of active customer master data.

Source:
Azure Database for PostgreSQL Flexible Server

Source Database:
{jdbc_database}

Source Table:
customer_source

Databricks Target:
{BRONZE_TABLE}

------------------------------------------------------------
SOURCE DATA
------------------------------------------------------------

Total source rows:
{source_count}

Active rows:
{active_count}

NULL-status rows:
{source_null_status_count}

Duplicate business keys:
{duplicate_key_count}

------------------------------------------------------------
INGESTION RESULT
------------------------------------------------------------

Extracted active rows:
{active_count}

Bronze rows:
{bronze_count}

NULL statuses in active extract:
{active_null_count}

Duplicate keys preserved:
{active_duplicate_count}

------------------------------------------------------------
EDGE CASES
------------------------------------------------------------

Active-only extraction:
{"PASS" if active_filter_passed else "FAIL"}

Zero matches:
{"PASS" if zero_count == 0 else "FAIL"}

Duplicate source keys:
{"PASS" if duplicate_key_count == 5 else "FAIL"}

NULL status:
{"PASS" if source_null_status_count == 10000 else "FAIL"}

Wrong credentials:
{"PASS" if wrong_credentials_failed else "FAIL"}

Database unavailable:
{"PASS" if database_unavailable_failed else "FAIL"}

Source-to-Bronze reconciliation:
{"PASS" if bronze_count == active_count else "FAIL"}

------------------------------------------------------------
TEST SUMMARY
------------------------------------------------------------

Passed:
{passed_count}

Failed:
{len(failed_tests)}

Total:
{total_tests}

Overall:
{"PASS" if len(failed_tests) == 0 else "FAIL"}

============================================================
"""

trainer_summary_file = (
    f"{REPORT_PATH}/"
    f"trainer_summary_{test_run_id}.txt"
)

dbutils.fs.put(
    trainer_summary_file,
    trainer_summary,
    overwrite=True
)


# ============================================================
# 21. DISPLAY FINAL RESULTS
# ============================================================

print("\n")
print("=" * 65)
print("DAY 4 FINAL TEST RESULTS")
print("=" * 65)

print(
    f"Passed : {passed_count}"
)

print(
    f"Failed : {len(failed_tests)}"
)

print(
    f"Total  : {total_tests}"
)

print()

if len(failed_tests) == 0:

    print(
        "ALL DAY 4 TESTS PASSED"
    )

else:

    print(
        "SOME DAY 4 TESTS FAILED"
    )

print()
print(
    f"Results table : {TEST_TABLE}"
)

print(
    f"Trainer report: {trainer_report_path}"
)

print(
    f"Summary file  : {trainer_summary_file}"
)

display(
    test_df
    .orderBy("test_name")
)